<a href="https://colab.research.google.com/github/rouuuuuuu/PFA/blob/main/DeBERTa_v3-%2B_MLP_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install scikit-learn
!pip install pandas numpy tqdm
!pip install torch
!pip install transformers scikit-learn pandas numpy torch tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DebertaV2Tokenizer, DebertaV2Model, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm import tqdm
from torch.optim import AdamW

# Vérifier si GPU dispo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Charger les données
df = pd.read_csv('/content/df_small_balanced_9000.csv')
print(df.columns)

# Colonnes importantes
text_column = 'CommentText'
num_columns = ['Likes', 'Replies']
label_column = 'Sentiment'

# Nettoyer NaN éventuels
df[text_column] = df[text_column].fillna("")
df[num_columns] = df[num_columns].fillna(0)

# Encoder les labels
le = LabelEncoder()
df['label'] = le.fit_transform(df[label_column])

# Standardiser les colonnes numériques
scaler = StandardScaler()
df[num_columns] = scaler.fit_transform(df[num_columns])

# Diviser en train/test
train_texts, test_texts, train_nums, test_nums, train_labels, test_labels = train_test_split(
    df[text_column].tolist(),
    df[num_columns].values,
    df['label'].values,
    test_size=0.2,
    random_state=42
)

# Tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained('microsoft/deberta-v3-base')


Using device: cuda
Index(['CommentID', 'VideoID', 'VideoTitle', 'AuthorName', 'AuthorChannelID',
       'CommentText', 'Sentiment', 'Likes', 'Replies', 'PublishedAt',
       'CountryCode', 'CategoryID'],
      dtype='object')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

In [ ]:
class CommentDataset(Dataset):
    def __init__(self, texts, nums, labels, tokenizer, max_len=256):
        self.texts = texts
        self.nums = nums
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        num = self.nums[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'num_feats': torch.tensor(num, dtype=torch.float),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Créer datasets
train_dataset = CommentDataset(train_texts, train_nums, train_labels, tokenizer)
test_dataset = CommentDataset(test_texts, test_nums, test_labels, tokenizer)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
class DebertaMLPClassifier(nn.Module):
    def __init__(self, num_numerical_feats, num_classes):
        super(DebertaMLPClassifier, self).__init__()
        self.deberta = DebertaV2Model.from_pretrained('microsoft/deberta-v3-base')
        hidden_size = self.deberta.config.hidden_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + num_numerical_feats, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, num_feats):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token

        combined = torch.cat((cls_output, num_feats), dim=1)
        logits = self.mlp(combined)
        return logits

num_classes = len(le.classes_)
model = DebertaMLPClassifier(num_numerical_feats=len(num_columns), num_classes=num_classes).to(device)


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

optimizer = AdamW(model.parameters(), lr=1e-5)
num_epochs = 30
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
criterion = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        num_feats = batch['num_feats'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, num_feats)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} finished. Avg Train Loss: {avg_loss:.4f}")

    # Évaluation à la fin de l'époque
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            num_feats = batch['num_feats'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask, num_feats)
            probs = nn.functional.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Pour AUC : on stocke les probabilités de la classe 1 (ou toutes les classes si multi-class)
            if probs.shape[1] == 2:
                all_probs.extend(probs[:, 1].cpu().numpy())
            else:
                all_probs.extend(probs.cpu().numpy())

    # Calcul métriques
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    try:
        if probs.shape[1] == 2:
            auc = roc_auc_score(all_labels, all_probs)
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    except:
        auc = float('nan')  # si AUC échoue (ex : une seule classe présente)

    print(f"Evaluation after Epoch {epoch+1}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-score:  {f1:.4f}")
    print(f"  AUC:       {auc:.4f}\n")



Epoch 1: 100%|██████████| 450/450 [07:01<00:00,  1.07it/s, loss=0.155]



Epoch 1 finished. Avg Train Loss: 0.6273
Evaluation after Epoch 1:
  Accuracy:  0.7978
  Precision: 0.8130
  Recall:    0.7978
  F1-score:  0.7969
  AUC:       0.9409



Epoch 2: 100%|██████████| 450/450 [07:00<00:00,  1.07it/s, loss=0.128]



Epoch 2 finished. Avg Train Loss: 0.4090
Evaluation after Epoch 2:
  Accuracy:  0.8361
  Precision: 0.8406
  Recall:    0.8361
  F1-score:  0.8375
  AUC:       0.9514



Epoch 3: 100%|██████████| 450/450 [07:00<00:00,  1.07it/s, loss=0.535]


...


Epoch 28 finished. Avg Train Loss: 0.0041
Evaluation after Epoch 28:
  Accuracy:  0.8344
  Precision: 0.8332
  Recall:    0.8344
  F1-score:  0.8336
  AUC:       0.9315



Epoch 29: 100%|██████████| 450/450 [07:01<00:00,  1.07it/s, loss=3.43e-5]



Epoch 29 finished. Avg Train Loss: 0.0054
Evaluation after Epoch 29:
  Accuracy:  0.8339
  Precision: 0.8331
  Recall:    0.8339
  F1-score:  0.8333
  AUC:       0.9336



Epoch 30: 100%|██████████| 450/450 [07:00<00:00,  1.07it/s, loss=3.36e-5]



Epoch 30 finished. Avg Train Loss: 0.0052
Evaluation after Epoch 30:
  Accuracy:  0.8344
  Precision: 0.8336
  Recall:    0.8344
  F1-score:  0.8338
  AUC:       0.9333

